# 08 · Recommender baselines (Phase 6)
Photo → top search match → **what do people buy next?**  Before any neural model, two simple opponents:
- **Popularity**: last week's 12 best-sellers, the same for everyone (ignores the input)
- **Co-purchase**: "after buying A, people most often bought B" (counting, no learning)

Scored on the **val week (9–15 Sep 2020)** with **MAP@12** (official H&M metric), twice:
full purchase history, and **last item only** (= the SnapShop photo case). Test week stays untouched.

In [1]:
from collections import Counter, defaultdict
from datetime import date

import numpy as np
import polars as pl

REC = "../data/processed/rec"
vocab = pl.read_parquet(f"{REC}/vocab.parquet")
idx2art = dict(zip(vocab["idx"].to_list(), vocab["article_id"].to_list()))   # item number -> article id
train = pl.read_parquet(f"{REC}/train_seqs.parquet")
val = pl.read_parquet(f"{REC}/val.parquet")
print(f"vocab {len(vocab):,} | train sequences {len(train):,} | val customers {len(val):,}")

vocab 29,881 | train sequences 656,182 | val customers 72,019


## The score: MAP@12
Per customer: 12 recommendations vs what they actually bought. Each hit scores *precision at that rank*
(hit at rank 2 → 1/2, a 2nd hit at rank 4 → 2/4), averaged → AP. MAP@12 = mean AP over all customers.

In [2]:
def apk(actual, predicted, k=12):
    """Average precision at k for one customer (official H&M Kaggle metric)."""
    if not actual:
        return 0.0
    hits, score = 0, 0.0
    for i, p in enumerate(predicted[:k]):
        if p in actual and p not in predicted[:i]:
            hits += 1
            score += hits / (i + 1)
    return score / min(len(actual), k)

def evaluate(df, recommend, last_only=False):
    """MAP@12 over all customers. last_only=True: the model sees only the LAST item (= the photo case)."""
    scores = []
    for targets, hist in zip(df["targets"].to_list(), df["history"].to_list()):
        h = hist or []
        if last_only:
            h = h[-1:]
        scores.append(apk(set(targets), recommend(h)))
    return float(np.mean(scores))

## Baseline 1: popularity (last training week's 12 best-sellers, no peeking at the val week)

In [3]:
tx = pl.scan_parquet("../data/processed/transactions.parquet")
pop = (tx.filter(pl.col("t_dat").is_between(date(2020, 9, 2), date(2020, 9, 8)))
         .group_by("article_id").len()
         .sort("len", descending=True)
         .head(12).collect()["article_id"].to_list())

def popularity(history):
    return pop                    # same 12 best-sellers for everyone; ignores the input

pop

['0915526001',
 '0751471043',
 '0751471001',
 '0706016001',
 '0919365008',
 '0915529003',
 '0918292001',
 '0863595006',
 '0896152002',
 '0448509014',
 '0909916001',
 '0762846031']

## Baseline 2: co-purchase
`co[a][b]` = how often **b** was bought within the next 3 purchases after **a** (all 656k training sequences).
Recommend from the customer's last 5 items (most recent counts most). No history → popularity.

In [4]:
co = defaultdict(Counter)
for items in train["items"].to_list():
    for i, a in enumerate(items):
        for b in items[i + 1:i + 4]:            # the next 3 purchases after a
            if b != a:
                co[a][b] += 1

def copurchase(history):
    if not history:
        return pop                                  # no history -> popularity (cold start)
    scores = Counter()
    for w, a in enumerate(reversed(history[-5:])):  # last 5 items; the most recent counts most
        for b, c in co[a].most_common(50):
            scores[b] += c / (w + 1)
    recs = [idx2art[b] for b, _ in scores.most_common(12)]
    return (recs + [p for p in pop if p not in recs])[:12]   # top up with popular items if < 12

results = {
    "popularity": (evaluate(val, popularity), evaluate(val, popularity, last_only=True)),
    "co-purchase": (evaluate(val, copurchase), evaluate(val, copurchase, last_only=True)),
}
pl.DataFrame([{"model": m, "MAP@12 full history": a, "MAP@12 last item only": b}
              for m, (a, b) in results.items()])

model,MAP@12 full history,MAP@12 last item only
str,f64,f64
"""popularity""",0.006604,0.006604
"""co-purchase""",0.011549,0.009711
